In [0]:
hchb_cash_collection=dbutils.widgets.get("hchb_cash_collection")
cash_collection=dbutils.widgets.get("cash_collection")
cash_unpivot=dbutils.widgets.get("cash_unpivot")
hchb_cash=dbutils.widgets.get("hchb_cash")
date_path=dbutils.widgets.get("date_path")
agencies=dbutils.widgets.get("agencies")
officemapping=dbutils.widgets.get("officemapping")
yrinvo=dbutils.widgets.get("yrinvo")
client_episodes_all=dbutils.widgets.get("client_episodes_all")
payerdimension=dbutils.widgets.get("payerdimension")
office=dbutils.widgets.get("office")
cubeserviceofficetxnsourcesystem=dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
client=dbutils.widgets.get("client")
paymentstype=dbutils.widgets.get("paymenttype")
paymentsdetails=dbutils.widgets.get("paymentsdetails")
datedimension=dbutils.widgets.get("datedimension")

In [0]:
spark.sql(
    f"""
TRUNCATE TABLE {cash_unpivot};
 """
) 

spark.sql(
    f"""
INSERT INTO {cash_unpivot} (
        company,
        agency,
        branch,
        client_name,
        medical_record_number,
        payor_types,
        payor_source,
        invoice,
        deposit_date,
        deposit,
        check,
        date_entered,
        month_end_close_reporting_period,
        psid,
        paymenttype,
        cashcollected
    )
SELECT 
    company,
    agency,
    branch,
    client_name,
    medical_record_number,
    payor_types,
    payor_source,
    invoice__,
    deposit_date,
    deposit__,
    check__,
    date_entered,
    month_end_close_reporting_period,
    psid,
    paymenttype,
    cashcollected
FROM (
    SELECT 
        company,
        agency,
        branch,
        client_name,
        medical_record_number,
        payor_types,
        payor_source,
        invoice__,
        rap_payment,
        final_payment,
        other_payment,
        CASE 
            WHEN (deposit__ NOT LIKE '%R%' AND refund_payment > '0.00') 
            THEN '0.00' 
            ELSE refund_payment 
        END AS refund_payment,
        unapplied_cash,
        deposit_date,
        deposit__,
        check__,
        date_entered,
        month_end_close_reporting_period,
        psid
    FROM {hchb_cash}
) c
UNPIVOT (
    cashcollected FOR paymenttype IN (
        rap_payment,
        final_payment,
        other_payment,
        refund_payment,
        unapplied_cash
    )
) AS u
WHERE cashcollected <> '0.00';
 """
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP view HCHBcash_temp AS
SELECT 
    'HCHB' AS SourceSystem,
    CASE 
        WHEN DAYOFWEEK(date_entered) = 1 THEN date_entered
        WHEN DAYOFWEEK(date_entered) = 2 THEN DATE_ADD(date_entered, -1)
        WHEN DAYOFWEEK(date_entered) = 3 THEN DATE_ADD(date_entered, -2)
        WHEN DAYOFWEEK(date_entered) = 4 THEN DATE_ADD(date_entered, -3)
        WHEN DAYOFWEEK(date_entered) = 5 THEN DATE_ADD(date_entered, 3)
        WHEN DAYOFWEEK(date_entered) = 6 THEN DATE_ADD(date_entered, 2)
        WHEN DAYOFWEEK(date_entered) = 7 THEN DATE_ADD(date_entered, 1)
    END AS `Reporting Week Ending Date`,
    company,
    agency,
    agency_id,
    CASE 
        WHEN branch RLIKE '[A-Z]' THEN OM.TargetOfficeNumber
        ELSE C.branch 
    END AS TargetOfficeNumber,
    branch,
    client_name,
    CltID,
    Payor_Types,
    Payor_Source,
    Invoice,
    CASE 
        WHEN d.DateKey IS NULL THEN '-1' 
        ELSE d.DateKey 
    END AS `Deposit Date Key`,
    Deposit,
    Check,
    CASE 
        WHEN dd.DateKey IS NULL THEN '-1' 
        ELSE dd.DateKey 
    END AS `Posted Date Key`,
    CASE 
        WHEN pd.PayerKey IS NULL THEN '-1' 
        ELSE pd.PayerKey 
    END AS `Payor Key`,
    CashCollected,
    PaymentType
FROM {cash_unpivot}  C
LEFT JOIN {date_path} D 
    ON C.Deposit_Date = D.CalendarDate
LEFT JOIN {date_path} DD 
    ON DD.CalendarDate = C.date_entered
LEFT JOIN {agencies} h 
    ON c.agency = h.agency_name
LEFT JOIN {officemapping} OM 
    ON OM.SourceOfficeCode = C.branch
LEFT JOIN {yrinvo} Y 
    ON Y.InvNum = C.Invoice
LEFT JOIN {payerdimension} PD  
    ON PD.PayerID = CAST(C.psid AS STRING);
 """
) 


spark.sql(
    f"""
CREATE OR REPLACE TEMPORARY VIEW ReportingWeekEndingDate_view AS
SELECT MAX(`Reporting Week Ending Date`) AS MaxReportingDate
FROM HCHBcash_temp;
 """
) 


spark.sql(
    f"""
INSERT INTO {paymentsdetails} (
    batchid,
    type,
    bank,
    batchnumber,
    checkid,
    agencyid,
    depositid,
    sourcesystem,
    landing_update_datetime,
    landing_update_by
)
SELECT DISTINCT 
    NULL AS batchid,
    NULL AS type,
    NULL AS bank,
    NULL AS batchnumber,
    T.Check AS checkid,
    T.agency_id AS agencyid,
    T.Deposit AS depositid,
    'HCHB' AS sourcesystem,
    current_timestamp() AS landing_update_datetime,
    'SYSTEM' AS landing_update_by
FROM HCHBcash_temp T
CROSS JOIN ReportingWeekEndingDate_view R
WHERE T.`Reporting Week Ending Date` = R.MaxReportingDate
  AND NOT EXISTS (
        SELECT 1 
        FROM {paymentsdetails} P 
        WHERE COALESCE(CAST(P.agencyid AS STRING), 'NULL') = COALESCE(CAST(T.agency_id AS STRING), 'NULL')
          AND COALESCE(P.checkid, 'NULL') = COALESCE(T.Check, 'NULL')
          AND COALESCE(P.depositid, 'NULL') = COALESCE(T.Deposit, 'NULL')
  );
"""
)




spark.sql(
    f"""
CREATE OR REPLACE TEMPORARY VIEW ClientIDs_temp AS
SELECT 
    SourceSystem,
    `Reporting Week Ending Date`,
    TargetOfficeNumber,
    branch,
    CltID,
    agency_id,
    Invoice,
    `Deposit Date Key`,
    Deposit,
    Check,
    `Posted Date Key`,
    `Payor Key`,
    CashCollected,
    PaymentType,
    epi_id,
    epi_paid
FROM (
    SELECT 
        t.*,
        e.epi_id,
        e.epi_paid
    FROM HCHBcash_temp t
    LEFT JOIN (
        SELECT 
            MAX(epi_id) AS epi_id,
            epi_paid,
            epi_branchcode
        FROM {client_episodes_all}
        GROUP BY epi_paid, epi_branchcode
    ) e 
        ON CAST(t.CltID AS STRING) = CAST(e.epi_paid AS STRING) 
        AND e.epi_branchcode = t.branch
) A;

 """
) 

spark.sql(
    f"""

CREATE OR REPLACE TEMPORARY VIEW FinalData_temp AS
SELECT 
    t.*,
    CASE 
        WHEN o.OfficeKey IS NULL THEN -1 
        ELSE o.OfficeKey 
    END AS `Office Key`,
    dad.DateKey AS `Reporting Week Ending DateKey`,
    SS.SourceSystemKey AS `Source System Key`,
    CASE 
        WHEN c.ClientKey IS NULL THEN -1 
        ELSE C.ClientKey 
    END AS `Client Key`,
    P.PaymentDetailKey,
    PT.PaymentTypeKey
FROM ClientIDs_temp t
LEFT JOIN {office} O
    ON O.OfficeNumber = t.TargetOfficeNumber
LEFT JOIN {datedimension} DAD
    ON DAD.CalendarDate = t.`Reporting Week Ending Date`
LEFT JOIN {cubeserviceofficetxnsourcesystem} SS
    ON SS.SourceSystemName = t.SourceSystem
LEFT JOIN {paymentsdetails} P
    ON COALESCE(CAST(P.AgencyID AS STRING), 'NULL') = COALESCE(CAST(T.agency_id AS STRING), 'NULL')
    AND COALESCE(P.CheckID, 'NULL') = COALESCE(T.Check, 'NULL')
    AND P.DepositID = T.Deposit
    AND P.SourceSystem = 'HCHB'
LEFT JOIN {paymentstype} PT
    ON PT.PaymentTypeDescription = T.PaymentType
LEFT JOIN {client} C
    ON CAST(C.SourceSystemId AS STRING) = CAST(t.epi_id AS STRING)
    AND c.OfficeNumber = t.TargetOfficeNumber;

 """
) 


In [0]:
spark.sql(
    f"""
TRUNCATE TABLE {hchb_cash_collection};

 """
) 



spark.sql(
    f"""
INSERT INTO {hchb_cash_collection} (
    sourcesystem,
    reporting_week_ending_date,
    targetofficenumber,
    branch,
    cltid,
    agency_id,
    invoice,
    deposit_date_key,
    deposit,
    check,
    posted_date_key,
    payor_key,
    cashcollected,
    paymenttype,
    epi_id,
    epi_paid,
    office_key,
    reporting_week_ending_datekey,
    source_system_key,
    client_key,
    paymentdetailkey,
    paymenttypekey
)
SELECT 
    SourceSystem,
    `Reporting Week Ending Date` AS reporting_week_ending_date,
    TargetOfficeNumber,
    branch,
    CltID,
    agency_id,
    Invoice AS invoice,
    `Deposit Date Key` AS deposit_date_key,
    Deposit AS deposit,
    Check AS check,
    `Posted Date Key` AS posted_date_key,
    `Payor Key` AS payor_key,
    CashCollected,
    PaymentType,
    epi_id,
    epi_paid,
    `Office Key` AS office_key,
    `Reporting Week Ending DateKey` AS reporting_week_ending_datekey,
    `Source System Key` AS source_system_key,
    `Client Key` AS client_key,
    PaymentDetailKey,
    PaymentTypeKey
FROM FinalData_temp;
"""
)


In [0]:
spark.sql(
    f"""
    delete from {cash_collection} where source_system_key=6
    """
)

In [0]:
spark.sql(
    f"""
INSERT INTO {cash_collection} (
    reporting_week_ending_date_key,
    posted_date_key,
    deposit_date_key,
    source_system_key,
    office_key,
    payor_key,
    client_key,
    payment_type_key,
    payment_detail_key,
    invoice_number,
    cash_collected 
)
SELECT 
    CAST(Reporting_Week_Ending_DateKey AS INT)        AS reporting_week_ending_date_key,
    CAST(Posted_Date_Key AS INT)                      AS posted_date_key,
    CAST(Deposit_Date_Key AS INT)                     AS deposit_date_key,
    CAST(Source_System_Key AS TINYINT)                AS source_system_key,
    CAST(Office_Key AS INT)                           AS office_key,
    CAST(Payor_Key AS INT)                            AS payor_key,
    CAST(Client_Key AS INT)                           AS client_key,
    CAST(PaymentTypeKey AS INT)                       AS payment_type_key,
    CAST(PaymentDetailKey AS INT)                     AS payment_detail_key,
    CAST(Invoice AS STRING)                           AS invoice_number,
    CAST(CashCollected AS DOUBLE)                     AS cash_collected 
FROM {hchb_cash_collection}
"""
)
